# Clase 099 — Detección de anomalías: Isolation Forest, LOF, One-Class SVM

Detectar outliers en datos sin etiquetas eligiendo entre **Isolation Forest**, **LOF** y **One-Class SVM** según la geometría, y distinguiendo *outlier detection* de *novelty detection*.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Isolation Forest baseline

2 blobs + 5% de outliers uniformes. Isolation Forest aísla anomalías con menos splits. Salida: `+1` inlier / `-1` outlier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest

np.random.seed(42)
rng = np.random.default_rng(42)

X_norm, _ = make_blobs(n_samples=950, centers=2, cluster_std=1.0, random_state=42)
X_out = rng.uniform(X_norm.min(axis=0) - 3, X_norm.max(axis=0) + 3, size=(50, 2))
X = np.vstack([X_norm, X_out])
es_outlier = np.r_[np.zeros(950, bool), np.ones(50, bool)]

iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=1).fit(X)
pred = iso.predict(X)  # -1 outlier, +1 inlier
recall = ((pred == -1) & es_outlier).sum() / es_outlier.sum()
print(f"outliers reales: {es_outlier.sum()} | detectados: {int((pred == -1).sum())}")
print(f"recall: {recall:.2%}")
assert recall >= 0.6, "IsolationForest deberia recuperar buena parte de los outliers"

plt.figure(figsize=(7, 5))
plt.scatter(X[pred == 1, 0], X[pred == 1, 1], c="#37a", s=8, label="inlier")
plt.scatter(X[pred == -1, 0], X[pred == -1, 1], c="#c33", s=30, marker="x", label="outlier")
plt.legend(); plt.title("Isolation Forest (contamination=0.05)")
plt.tight_layout(); plt.show()

## 2. LOF para anomalías locales

Metemos un outlier *dentro* de un cluster denso. LOF compara densidad local con la de los vecinos; Isolation Forest (global) suele fallar aquí.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

X_local = np.vstack([X_norm, [X_norm[:475].mean(axis=0) + [0.3, 0.3]]])  # punto raro pero interno
idx_local = len(X_local) - 1

lof = LocalOutlierFactor(n_neighbors=20)
pred_lof = lof.fit_predict(X_local)
print("LOF marca el punto interno como outlier:", pred_lof[idx_local] == -1)

iso2 = IsolationForest(contamination=0.01, random_state=42, n_jobs=1).fit(X_local)
print("IsolationForest lo marca:", iso2.predict(X_local)[idx_local] == -1)
print("(LOF es mejor para anomalias LOCALES dentro de un cluster denso)")

plt.figure(figsize=(7, 5))
plt.scatter(X_local[:-1, 0], X_local[:-1, 1], c="#37a", s=8)
plt.scatter(X_local[idx_local, 0], X_local[idx_local, 1], c="#c33", s=120, marker="*", label="anomalia local")
plt.legend(); plt.title("Anomalia local: caso de LOF")
plt.tight_layout(); plt.show()

## 3. One-Class SVM y el escalado

El kernel RBF es ultra sensible a la escala: sin `StandardScaler` marca casi todo mal.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

X_skew = X.copy()
X_skew[:, 0] *= 50  # una feature domina

oc_sin = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit_predict(X_skew)
oc_con = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit_predict(
    StandardScaler().fit_transform(X_skew))

rec_sin = ((oc_sin == -1) & es_outlier).sum() / es_outlier.sum()
rec_con = ((oc_con == -1) & es_outlier).sum() / es_outlier.sum()
print(f"recall sin escalar: {rec_sin:.2%}")
print(f"recall con escalar: {rec_con:.2%}")
assert rec_con >= rec_sin, "escalar deberia ayudar (o al menos no empeorar) al One-Class SVM"
print("El kernel RBF exige estandarizar antes.")

## 4. Top-k con `score_samples`

`score_samples` da un score continuo (más alto = más normal). Ordenamos ascendente y sacamos los 10 más anómalos.

In [ ]:
scores = iso.score_samples(X)  # mas bajo = mas anomalo
top10 = np.argsort(scores)[:10]
print("scores de los 10 puntos mas anomalos:", scores[top10].round(3))
print("cuantos de esos 10 son outliers reales:", int(es_outlier[top10].sum()), "/ 10")
assert es_outlier[top10].sum() >= 7, "el top-10 por score deberia estar dominado por outliers reales"

plt.figure(figsize=(7, 4))
plt.hist(scores[~es_outlier], bins=40, alpha=0.7, label="normales", color="#37a")
plt.hist(scores[es_outlier], bins=40, alpha=0.7, label="outliers", color="#c33")
plt.xlabel("score_samples (mas bajo = mas anomalo)"); plt.ylabel("frecuencia")
plt.title("Distribucion de scores: los outliers caen a la izquierda")
plt.legend(); plt.tight_layout(); plt.show()

## 5. Comparativa de los tres detectores

Mismos datos escalados, misma tarea: comparamos recall de outliers y tiempo de entrenamiento.

In [ ]:
import time
from sklearn.neighbors import LocalOutlierFactor

Xs = StandardScaler().fit_transform(X)

def evaluar(nombre, fit_pred):
    t0 = time.perf_counter()
    pred = fit_pred(Xs)
    dt = time.perf_counter() - t0
    rec = ((pred == -1) & es_outlier).sum() / es_outlier.sum()
    return nombre, rec, dt

resultados = [
    evaluar("IsolationForest", lambda X: IsolationForest(contamination=0.05, random_state=42, n_jobs=1).fit_predict(X)),
    evaluar("LOF",             lambda X: LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(X)),
    evaluar("OneClassSVM",     lambda X: OneClassSVM(kernel="rbf", nu=0.05, gamma="scale").fit_predict(X)),
]
print(f"{'modelo':>16} {'recall':>8} {'tiempo(ms)':>12}")
for nombre, rec, dt in resultados:
    print(f"{nombre:>16} {rec:>7.1%} {dt*1000:>11.1f}")
print("\nIsolation Forest: default solido y rapido. LOF: anomalias locales. OneClassSVM: frontera no lineal, lento.")

## Ejercicios

1. Generá 2 blobs + 5% de outliers, ajustá `IsolationForest(contamination=0.05)` y graficá inliers vs outliers.
2. Meté un outlier *dentro* de un blob y comparó `IsolationForest` vs `LocalOutlierFactor(n_neighbors=20)`.
3. Entrená `OneClassSVM(kernel='rbf', nu=0.05)` con y sin `StandardScaler` y compará el recall.
4. Usá `score_samples`, ordená ascendente y devolvé los 10 puntos más anómalos.

## Conclusiones

- **Isolation Forest** es el default sólido: escala bien, pocos hiperparámetros, robusto en alta dimensión.
- **LOF** detecta anomalías **locales** (un raro dentro de un cluster denso) que los métodos globales pierden.
- **One-Class SVM** aprende una frontera no lineal pero es **ultra sensible a la escala**: estandarizá siempre.
- Preferí `score_samples` + top-k o ROC-AUC sobre `accuracy`: con clases desbalanceadas el accuracy engaña.

## ✅ Soluciones de los ejercicios

Cinco ejercicios de detección de anomalías: Isolation Forest, LOF, One-Class SVM, ranking por `score_samples` y evaluación con labels. El README usa `fetch_kddcup99` (requiere internet); lo reemplazamos por un **dataset sintético etiquetado** (normales + anomalías inyectadas) para correr offline. `n_jobs=1`.

**Ejercicio 1 — Isolation Forest baseline.** 2 blobs + 5% de outliers uniformes; `contamination=0.05`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
Xin, _ = make_blobs(n_samples=475, centers=[[0, 0], [6, 6]], cluster_std=0.7, random_state=42)
Xout = rng.uniform(-6, 12, (25, 2))
X = np.vstack([Xin, Xout])
iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=1).fit(X)
pred = iso.predict(X)  # 1 inlier, -1 outlier
plt.figure(figsize=(6, 5))
plt.scatter(X[pred == 1, 0], X[pred == 1, 1], c='#37a', s=10, label='inlier')
plt.scatter(X[pred == -1, 0], X[pred == -1, 1], c='#c33', s=25, marker='x', label='outlier')
plt.legend(); plt.title('Isolation Forest'); plt.tight_layout(); plt.show()
print('outliers detectados:', int((pred == -1).sum()))

**Ejercicio 2 — LOF y anomalías locales.** Metemos un punto *dentro* de un blob: LOF (densidad local) lo detecta mejor que Isolation Forest.

In [ ]:
Xloc = np.vstack([Xin, [[6, 6.0]], [[0.2, 0.2]]])  # dos puntos en zonas densas... uno normal
# anomalia local: un punto en un hueco de baja densidad cerca del centro
Xloc = np.vstack([Xin, [[3, 3]]])  # entre los dos blobs, densidad local baja
lof = LocalOutlierFactor(n_neighbors=20)
lab_lof = lof.fit_predict(Xloc)
iso2 = IsolationForest(contamination=0.02, random_state=42, n_jobs=1).fit(Xloc)
lab_iso = iso2.predict(Xloc)
i = len(Xloc) - 1  # el punto sospechoso
print(f'punto local [3,3] -> LOF: {"OUTLIER" if lab_lof[i]==-1 else "inlier"} | '
      f'IsolationForest: {"OUTLIER" if lab_iso[i]==-1 else "inlier"}')
print('LOF compara la densidad de un punto con la de sus vecinos: agarra anomalias locales.')

**Ejercicio 3 — One-Class SVM y escalado.** Con RBF, el escalado cambia la frontera: sin escalar, `gamma` no ve bien las escalas relativas.

In [ ]:
Xsk = X.copy(); Xsk[:, 1] *= 20  # descompensar escalas
oc_raw = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale').fit(Xsk)
oc_sc = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale').fit(StandardScaler().fit_transform(Xsk))
print('sin escalar -> outliers:', int((oc_raw.predict(Xsk) == -1).sum()))
print('escalado    -> outliers:', int((oc_sc.predict(StandardScaler().fit_transform(Xsk)) == -1).sum()))
print('One-Class SVM es sensible a la escala: estandarizar es casi obligatorio con RBF.')

**Ejercicio 4 — Top-k con `score_samples`.** Ordenamos ascendente (más bajo = más anómalo) y devolvemos los 10 puntos más anómalos.

In [ ]:
scores = iso.score_samples(X)  # mas bajo = mas anomalo
top10 = np.argsort(scores)[:10]
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c='#ccc', s=10)
plt.scatter(X[top10, 0], X[top10, 1], c='#c33', s=60, marker='*', label='top-10 anomalos')
plt.legend(); plt.title('Top-10 por score_samples'); plt.tight_layout(); plt.show()
print('scores de los 10 mas anomalos:', np.round(scores[top10], 3))

**Ejercicio 5 — Evaluación con labels (ROC-AUC).** En vez de `fetch_kddcup99` (necesita internet) usamos un dataset sintético con labels: entrenamos Isolation Forest **sin** labels y medimos ROC-AUC contra la verdad.

In [ ]:
# dataset etiquetado offline: 950 normales + 50 anomalias (2 features informativas)
Xn = rng.normal(0, 1, (950, 6))
Xa = rng.normal(4, 1.5, (50, 6))  # anomalias desplazadas
Xk = np.vstack([Xn, Xa])
yk = np.hstack([np.zeros(950), np.ones(50)])  # 1 = anomalia
iso_k = IsolationForest(contamination=0.05, random_state=42, n_jobs=1).fit(Xk)
# score_samples: mas bajo = mas anomalo -> negamos para que mayor = mas anomalo
auc = roc_auc_score(yk, -iso_k.score_samples(Xk))
print(f'ROC-AUC (Isolation Forest, sin usar labels al entrenar): {auc:.4f}')
assert auc > 0.8, 'deberia distinguir bien anomalias claramente desplazadas'
print('OK: el modelo no supervisado separa normal vs anomalia (AUC alto).')